# Fine-tune Gemma 4 E2B for Text-to-SQL (QLoRA, T4-friendly)

Runtime > Change runtime type > **T4 GPU**, then run all cells top to bottom.

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets evaluate peft trl tensorboard

In [ ]:
import torch
from huggingface_hub import notebook_login

print("GPU:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

notebook_login()

## 1. Dataset

In [ ]:
from datasets import load_dataset

system_message = """You are an expert SQL developer. Your task is to generate accurate SQL queries based on the text or natural-language instructions provided by the user. Carefully analyze the requirements, write the appropriate SQL query, and thoroughly review and validate the query to ensure it is correct, efficient, and error-free before providing the final output."""

user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""


def create_conversation(sample):
    return {
        "prompt": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt.format(question=sample["sql_prompt"], context=sample["sql_context"])},
        ],
        "completion": [
            {"role": "assistant", "content": sample["sql"]},
        ],
    }


dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.select(range(1250))
dataset = dataset.map(create_conversation, remove_columns=dataset.features)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)

for item in dataset["train"][0]["prompt"] + dataset["train"][0]["completion"]:
    print(item)

## 2. Model (4-bit, fp16 compute)

T4 has no native bf16, so everything is made fp16. If you are on an L4/A100, you can switch `dtype` to `torch.bfloat16` here and set `bf16=True, fp16=False` in the training config.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-4-E2B"
dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=dtype,
    device_map="auto",
)

# Gemma checkpoints are stored in bf16: convert any leftover bf16 weights/buffers to fp16
for p in model.parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for b in model.buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Fallback chat template if the tokenizer does not ship one
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
        "{% endfor %}"
    )

## 3. LoRA + training config

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# No modules_to_save / ensure_weight_tying: avoids huge trainable embedding copies (OOM risk on T4)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

args = SFTConfig(
    output_dir="gemma-text-to-sql",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    fp16=True,
    bf16=False,
    push_to_hub=False,
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

# LoRA params must be fp32 for the fp16 grad scaler
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

In [ ]:
# Sanity check: trainable params should be float32 only
from collections import Counter

print("Model parameter dtypes:")
print(Counter(str(p.dtype) for p in trainer.model.parameters()))

print("\nTrainable parameter dtypes:")
print(Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))

## 4. Train

In [ ]:
trainer.train()

## 5. Save the adapter and test it

In [ ]:
trainer.save_model("gemma-text-to-sql")
tokenizer.save_pretrained("gemma-text-to-sql")

# Optional: push the adapter to your Hugging Face account
# trainer.push_to_hub()

In [ ]:
# Quick inference test on a held-out example
sample = dataset["test"][0]
messages = sample["prompt"]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(trainer.model.device)

trainer.model.eval()
with torch.no_grad():
    out = trainer.model.generate(**inputs, max_new_tokens=128, do_sample=False)

generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated:", generated)
print("Expected: ", sample["completion"][0]["content"])